<a href="https://colab.research.google.com/github/luca-fce/siniestros-futbol-caba/blob/main/notebook/3_1_river_boca.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Partidos de River Plate y Boca Juniors (2021–2025)

En esta parte juntamos las planillas anuales de Boca Juniors y River Plate (fuente: FBref) en una sola tabla, para que la investigación sea más sencilla. Después exploramos su calidad y la preparamos para cruzarla con los siniestros viales.

> *Uso de IA:* notebook armado con asistencia de Claude (Anthropic). Prompt: «Uní los CSV anuales de River y Boca 2021–2025 de FBref en una tabla, explorá su calidad y dejá la hora de inicio en hora argentina y el resultado final considerando los penales». El grupo revisó el código y las interpretaciones.

## 1. Carga de los datos

Hay un CSV por equipo y por año (10 archivos). Los leemos directo desde el repositorio del grupo y agregamos dos columnas para identificar cada fila: `equipo` y `temporada`.

In [1]:
import pandas as pd
import numpy as np

REPO = 'https://raw.githubusercontent.com/luca-fce/siniestros-futbol-caba/main/datos/Temporadas%20River%20-%20Boca/'
archivos = {'Boca Juniors': 'Boca%20Juniors/boca_{}.csv', 'River Plate': 'River%20Plate/river_{}.csv'}

lista = []
for equipo, patron in archivos.items():
    for anio in range(2021, 2026):
        df = pd.read_csv(REPO + patron.format(anio))
        df.insert(0, 'equipo', equipo)
        df.insert(1, 'temporada', anio)
        lista.append(df)
        print(f'{equipo} {anio}: {len(df)} partidos')

partidos_raw = pd.concat(lista, ignore_index=True)
print('\nTotal:', partidos_raw.shape)

Boca Juniors 2021: 48 partidos
Boca Juniors 2022: 52 partidos


Boca Juniors 2023: 54 partidos
Boca Juniors 2024: 53 partidos


Boca Juniors 2025: 42 partidos
River Plate 2021: 49 partidos


River Plate 2022: 50 partidos
River Plate 2023: 51 partidos


River Plate 2024: 54 partidos
River Plate 2025: 48 partidos

Total: (501, 20)


In [2]:
partidos_raw.head()

,equipo,temporada,Date,Time,Comp,Round,Day,Venue,Result,GF,GA,Opponent,Poss,Attendance,Captain,Formation,Opp Formation,Referee,Match Report,Notes
0,Boca Juniors,2021,2021-02-14,19:20,Copa de la Liga Profesional,Group stage,Sun,Home,D,2,2,Gimnasia–LP,67.0,NaN,Carlos Tevez,4-2-3-1,4-2-3-1,Pablo Echavarría,Match Report,NaN
1,Boca Juniors,2021,2021-02-21,21:30,Copa de la Liga Profesional,Group stage,Sun,Away,W,1,0,Newell's,53.0,NaN,Carlos Izquierdoz,4-4-2,4-3-3,Darío Herrera,Match Report,NaN
2,Boca Juniors,2021,2021-02-28,21:30,Copa de la Liga Profesional,Group stage,Sun,Home,D,1,1,Sarmiento–J,71.0,NaN,Carlos Izquierdoz,4-4-2,4-2-3-1,Francisco Oscar Lamolina,Match Report,NaN
3,Boca Juniors,2021,2021-03-07,21:30,Copa de la Liga Profesional,Group stage,Sun,Away,W,7,1,Vélez Sarsfield,41.0,NaN,Carlos Tevez,4-1-4-1,4-2-3-1,Diego Abal,Match Report,NaN
4,Boca Juniors,2021,2021-03-14,18:00,Copa de la Liga Profesional,Group stage,Sun,Home,D,1,1,River Plate,NaN,NaN,Carlos Tevez,3-4-1-2,4-3-3,Facundo Tello,Match Report,NaN


In [3]:
partidos_raw.tail()

,equipo,temporada,Date,Time,Comp,Round,Day,Venue,Result,GF,GA,Opponent,Poss,Attendance,Captain,Formation,Opp Formation,Referee,Match Report,Notes
496,River Plate,2025,2025-10-18,20:30,Liga Argentina,Torneo Clausura — Regular season,Sat,Away,W,2,0,Talleres–C,54.0,NaN,Franco Armani,4-3-1-2,4-3-3,Ariel Penel,Match Report,NaN
497,River Plate,2025,2025-11-02,20:30,Liga Argentina,Torneo Clausura — Regular season,Sun,Home,L,0,1,Gimnasia–LP,64.0,NaN,Franco Armani,4-2-3-1,4-2-3-1,Nazareno Arasa,Match Report,NaN
498,River Plate,2025,2025-11-09,16:30,Liga Argentina,Torneo Clausura — Regular season,Sun,Away,L,0,2,Boca Juniors,55.0,NaN,Franco Armani,3-5-2,4-2-2-2,Nicolás Ramírez,Match Report,NaN
499,River Plate,2025,2025-11-16,17:00,Liga Argentina,Torneo Clausura — Regular season,Sun,Away,D,0,0,Vélez Sarsfield,55.0,NaN,Enzo Pérez,4-1-3-2,4-2-3-1,Leandro Rey,Match Report,NaN
500,River Plate,2025,2025-11-24,19:15,Liga Argentina,Torneo Clausura — Round of 16,Mon,Away,L,2,3,Racing Club,47.0,NaN,Enzo Pérez,4-3-1-2,4-3-3,Facundo Tello,Match Report,NaN


## 2. Estructura y calidad

Revisamos tipos de dato, valores nulos, duplicados y las categorías principales.

In [4]:
partidos_raw.dtypes

equipo               str
temporada          int64
Date                 str
Time                 str
Comp                 str
Round                str
Day                  str
Venue                str
Result               str
GF                object
GA                object
Opponent             str
Poss             float64
Attendance       float64
Captain              str
Formation            str
Opp Formation        str
Referee              str
Match Report         str
Notes                str
dtype: object

In [5]:
print('Nulos por columna:')
print(partidos_raw.isnull().sum())
print('\nFilas duplicadas:', partidos_raw.duplicated().sum())
print('Partidos repetidos (equipo + fecha):', partidos_raw.duplicated(['equipo', 'Date']).sum())

Nulos por columna:
equipo             0
temporada          0
Date               0
Time               0
Comp               0
Round              0
Day                0
Venue              0
Result             0
GF                 0
GA                 0
Opponent           0
Poss               2
Attendance       466
Captain            0
Formation          0
Opp Formation      1
Referee           40
Match Report       0
Notes            458
dtype: int64

Filas duplicadas: 0
Partidos repetidos (equipo + fecha): 0


In [6]:
print(partidos_raw['Comp'].value_counts(), '\n')
print(partidos_raw['Venue'].value_counts(), '\n')
print(partidos_raw['Result'].value_counts())

Comp
Liga Argentina                 284
Copa de la Liga Profesional    122
Libertadores                    79
Sudamericana                    10
Club World Cup                   6
Name: count, dtype: int64 

Venue
Home       253
Away       242
Neutral      6
Name: count, dtype: int64 

Result
W    244
D    143
L    114
Name: count, dtype: int64


In [7]:
# Valores raros detectados
print('Horarios con paréntesis:', partidos_raw['Time'].str.contains(r'\(').sum(),
      '-> ej:', partidos_raw.loc[partidos_raw['Time'].str.contains(r'\('), 'Time'].unique()[:4])
print('Goles con paréntesis (penales):', partidos_raw['GF'].astype(str).str.contains(r'\(').sum(),
      '-> ej:', partidos_raw.loc[partidos_raw['GF'].astype(str).str.contains(r'\('), 'GF'].unique()[:4])
print('Rivales con código de país:', partidos_raw['Opponent'].str.match(r'^[a-z]{2} ').sum(),
      '-> ej:', partidos_raw.loc[partidos_raw['Opponent'].str.match(r'^[a-z]{2} '), 'Opponent'].unique()[:4])

Horarios con paréntesis: 29 -> ej: <StringArray>
['18:00 (19:00)', '19:30 (21:30)', '20:00 (21:00)', '19:00 (21:00)']
Length: 4, dtype: str
Goles con paréntesis (penales): 17 -> ej: ['1 (4)' '0 (2)' '0 (1)' '0 (6)']
Rivales con código de país: 95 -> ej: <StringArray>
['bo The Strongest', 'br Santos', 'ec Barcelona', 'br Atlético Mineiro']
Length: 4, dtype: str


**Hallazgos:**

- **501 partidos** (249 de Boca y 252 de River), sin filas duplicadas. Cada equipo juega unos 50 partidos oficiales por año.
- Las columnas con muchos nulos (`Attendance`, `Notes`) no son necesarias para la pregunta, salvo `Notes`, que indica alargue y penales.
- **Horarios:** FBref muestra la hora local del estadio. Cuando el partido se jugó en otro huso horario, agrega entre paréntesis la hora argentina (ej. `19:30 (21:30)` en Ecuador).
- **Goles:** en los partidos definidos por penales, `GF` y `GA` vienen como texto, por ejemplo `1 (4)`: goles en el partido y, entre paréntesis, penales convertidos. Por eso pandas no los lee como números.
- **Resultado:** FBref marca como empate (`D`) un partido definido por penales, aunque un equipo haya quedado eliminado.
- **Rivales internacionales** llevan un código de país adelante (`br Palmeiras`).
- **Cobertura:** solo están Liga, Copa de la Liga, Libertadores, Sudamericana y Mundial de Clubes. **No figuran la Copa Argentina ni las supercopas**, que también son oficiales. Lo informamos como limitación.

## 3. Transformaciones

1. **Hora argentina:** si hay paréntesis usamos esa hora; si no, la hora local ya es la argentina.
2. **Goles y penales:** separamos `1 (4)` en goles (`goles_a_favor`) y penales (`penales_a_favor`), y los pasamos a números.
3. **Resultado final:** para el hincha, quedar eliminado por penales es una derrota. Reclasificamos los partidos definidos por penales según quién los ganó.
4. **Fecha y hora de inicio y de fin:** el fin se estima en inicio + 1 h 55 min (90' + entretiempo + adicionado); con alargue se suman 30 min y con penales 15 min más.
5. **Rival sin código de país** y una marca para los **superclásicos**.

In [8]:
p = partidos_raw.copy()

# 1. Hora argentina
p['hora_arg'] = p['Time'].str.extract(r'\((\d{2}:\d{2})\)')[0].fillna(p['Time'].str[:5])

# 2. Goles y penales
for col, nombre in [('GF', 'a_favor'), ('GA', 'en_contra')]:
    partes = p[col].astype(str).str.extract(r'^\s*(\d+)\s*(?:\((\d+)\))?')
    p[f'goles_{nombre}'] = pd.to_numeric(partes[0]).astype('Int64')
    p[f'penales_{nombre}'] = pd.to_numeric(partes[1]).astype('Int64')
p['penales'] = p['penales_a_favor'].notna()
p['alargue'] = p['Notes'].fillna('').str.contains('extra time', case=False)

# 3. Resultado final (penales incluidos)
p['resultado'] = p['Result'].map({'W': 'Victoria', 'D': 'Empate', 'L': 'Derrota'})
p.loc[p['penales'] & (p['penales_a_favor'] > p['penales_en_contra']), 'resultado'] = 'Victoria'
p.loc[p['penales'] & (p['penales_a_favor'] < p['penales_en_contra']), 'resultado'] = 'Derrota'

# 4. Inicio y fin
p['inicio'] = pd.to_datetime(p['Date'] + ' ' + p['hora_arg'])
p['fin'] = (p['inicio'] + pd.Timedelta(minutes=115)
            + pd.to_timedelta(np.where(p['alargue'], 30, 0), unit='min')
            + pd.to_timedelta(np.where(p['penales'], 15, 0), unit='min'))

# 5. Rival y superclásico
p['rival'] = p['Opponent'].str.replace(r'^[a-z]{2} ', '', regex=True)
p['superclasico'] = p['rival'].isin(['River Plate', 'Boca Juniors'])

partidos = (p.rename(columns={'Comp': 'competencia', 'Round': 'instancia', 'Venue': 'condicion'})
             [['equipo', 'temporada', 'inicio', 'fin', 'hora_arg', 'competencia', 'instancia', 'condicion', 'rival',
               'goles_a_favor', 'goles_en_contra', 'penales', 'penales_a_favor', 'penales_en_contra',
               'alargue', 'resultado', 'superclasico']]
             .sort_values('inicio').reset_index(drop=True))

print('Partidos definidos por penales:', partidos['penales'].sum())
print('Cambiados de Empate a Victoria/Derrota:',
      ((p['Result'] == 'D') & p['penales']).sum())
partidos[partidos['penales']][['equipo', 'inicio', 'rival', 'goles_a_favor', 'goles_en_contra',
                               'penales_a_favor', 'penales_en_contra', 'resultado']]

Partidos definidos por penales: 17
Cambiados de Empate a Victoria/Derrota: 14


,equipo,inicio,rival,goles_a_favor,goles_en_contra,penales_a_favor,penales_en_contra,resultado
34,Boca Juniors,2021-05-16 17:30:00,River Plate,1,1,4,2,Victoria
35,River Plate,2021-05-16 17:30:00,Boca Juniors,1,1,2,4,Derrota
40,Boca Juniors,2021-05-31 15:00:00,Racing Club,0,0,2,4,Derrota
45,Boca Juniors,2021-07-20 19:15:00,Atlético Mineiro,0,0,1,3,Derrota
135,Boca Juniors,2022-05-14 17:00:00,Racing Club,0,0,6,5,Victoria
155,Boca Juniors,2022-07-05 21:30:00,Corinthians,0,0,5,6,Derrota
267,River Plate,2023-08-08 21:00:00,Internacional,1,2,8,9,Derrota
268,Boca Juniors,2023-08-09 21:00:00,Nacional,2,2,4,2,Victoria
274,Boca Juniors,2023-08-30 21:30:00,Racing Club,0,0,4,1,Victoria
286,Boca Juniors,2023-10-05 21:30:00,Palmeiras,1,1,4,2,Victoria


## 4. Resumen agrupado

In [9]:
resumen = (partidos.groupby(['equipo', 'resultado']).size().unstack()[['Victoria', 'Empate', 'Derrota']])
resumen['total'] = resumen.sum(axis=1)
resumen['% derrotas'] = (resumen['Derrota'] / resumen['total'] * 100).round(1)
resumen

resultado,Victoria,Empate,Derrota,total,% derrotas
equipo,,,,,
Boca Juniors,120,64,65,249,26.1
River Plate,130,65,57,252,22.6


In [10]:
pd.crosstab(partidos['temporada'], [partidos['equipo'], partidos['resultado']])

equipo    Boca Juniors                 River Plate                
resultado      Derrota Empate Victoria     Derrota Empate Victoria
temporada                                                         
2021                13     14       21          11     13       25
2022                11     12       29          13      9       28
2023                17     12       25          12      8       31
2024                13     16       24           9     19       26
2025                11     10       21          12     16       20

In [11]:
# ¿A qué hora terminan los partidos? (define dónde caen las ventanas de 12 h)
partidos['fin'].dt.hour.value_counts().sort_index()

fin
0       9
11      1
15      2
16     10
17     20
18     30
19     42
20     68
21     67
22    125
23    127
Name: count, dtype: int64

**Interpretación.** Boca y River ganan cerca de la mitad de sus partidos y pierden alrededor de uno de cada cuatro o cinco: en cinco años suman **122 derrotas**, suficientes para un análisis estadístico. La gran mayoría de los partidos termina de noche (entre las 20 y las 24 h), por lo que la ventana de 12 h posterior cubre sobre todo la noche y la madrugada, cuando la circulación es menor.

## 5. Exportación

Guardamos tres archivos: una planilla por equipo con sus cinco temporadas juntas y la tabla combinada de los dos equipos, que es la que se usa en el análisis. El análisis principal (`3_2_cruce_partidos_siniestros.ipynb`) aplica estas mismas reglas.

In [12]:
partidos[partidos['equipo'] == 'Boca Juniors'].to_csv('boca_2021_2025.csv', index=False)
partidos[partidos['equipo'] == 'River Plate'].to_csv('river_2021_2025.csv', index=False)
partidos.to_csv('river_boca_2021_2025.csv', index=False)

print('boca_2021_2025.csv:', (partidos['equipo'] == 'Boca Juniors').sum(), 'partidos')
print('river_2021_2025.csv:', (partidos['equipo'] == 'River Plate').sum(), 'partidos')
print('river_boca_2021_2025.csv:', len(partidos), 'partidos')

boca_2021_2025.csv: 249 partidos
river_2021_2025.csv: 252 partidos
river_boca_2021_2025.csv: 501 partidos
